# 02 — Nettoyage des données brutes

Ce notebook nettoie le dataset brut Kaggle sans jamais modifier le fichier source,
qui reste la vérité de référence immuable (voir `CLAUDE.md` et `Docs/DATA_SOURCES.md`).

**Entrée** : `data/raw/final_perfume_data.csv.zip`
(2191 parfums, colonnes `Name`, `Brand`, `Description`, `Notes`, `Image URL` —
confirmées lors de l'exploration, notebook 01)

**Sortie** : `data/processed/nayaar_clean.csv`

Ce notebook fait **uniquement du nettoyage** — pas de feature engineering
(saison, style, usage...), ce sera l'objet du notebook 03.

Étapes :
1. Chargement du dataset brut
2. Suppression des doublons (clé nom + marque normalisés)
3. Gestion des valeurs manquantes (stratégie documentée par colonne)
4. Normalisation des textes (casse, espaces, caractères spéciaux, encodage)
5. Normalisation des noms de marque
6. Parsing de la colonne `Notes` en liste (`notes_list`)
7. Rapport final avant / après
8. Export vers `data/processed/nayaar_clean.csv`


In [2]:
# Imports
import json
import re
import unicodedata

import pandas as pd

# Chemins relatifs : le notebook est exécuté depuis data/notebooks/
# (dossier courant = celui du notebook, comportement par défaut de Jupyter)
CHEMIN_DATASET_BRUT = "../raw/final_perfume_data.csv.zip"
CHEMIN_DATASET_PROPRE = "../processed/nayaar_clean.csv"

# Encodage confirmé lors de l'exploration (notebook 01) : le fichier n'est pas
# de l'UTF-8 valide (échec de décodage sur plusieurs octets). latin-1 est
# l'encodage qui permet de lire l'intégralité du fichier sans erreur.
ENCODAGE_BRUT = "latin-1"


In [3]:
# Rapport de nettoyage : on accumule ici chaque étape pour le résumé final
rapport_nettoyage = []


def enregistrer_etape(nom_etape, nb_lignes_avant, nb_lignes_apres, commentaire=""):
    """Ajoute une ligne au rapport de nettoyage pour garder une trace de chaque transformation."""
    rapport_nettoyage.append({
        "etape": nom_etape,
        "lignes_avant": nb_lignes_avant,
        "lignes_apres": nb_lignes_apres,
        "lignes_supprimees": nb_lignes_avant - nb_lignes_apres,
        "commentaire": commentaire,
    })


## 1. Chargement du dataset brut

In [4]:
def charger_dataset_brut(chemin_zip, encodage):
    """
    Charge le CSV brut directement depuis son archive zip (pandas décompresse
    automatiquement quand l'archive ne contient qu'un seul fichier), sans
    jamais extraire ni modifier le fichier source sur le disque.
    """
    df = pd.read_csv(chemin_zip, encoding=encodage)
    return df


df_brut = charger_dataset_brut(CHEMIN_DATASET_BRUT, ENCODAGE_BRUT)
nb_lignes_brutes = len(df_brut)

print(f"Dataset brut chargé : {nb_lignes_brutes} lignes, colonnes : {list(df_brut.columns)}")
df_brut.head()


Dataset brut chargé : 2191 lignes, colonnes : ['Name', 'Brand', 'Description', 'Notes', 'Image URL']


,Name,Brand,Description,Notes,Image URL
0,Tihota Eau de Parfum,Indult,"Rapa Nui for sugar, Tihota is, quite simply, ...","Vanilla bean, musks",https://static.luckyscent.com/images/products/...
1,Sola Parfum,Di Ser,A tribute to the expanse of space extending f...,"Lavender, Yuzu, Lemongrass, Magnolia, Geraniu...",https://static.luckyscent.com/images/products/...
2,Kagiroi Parfum,Di Ser,An aromatic ode to the ancient beauty of Japa...,"Green yuzu, green shikuwasa, sansho seed, cor...",https://static.luckyscent.com/images/products/...
3,Velvet Fantasy Eau de Parfum,Montale,Velvet Fantasy is a solar fragrance where cit...,"tangerine, pink pepper, black coffee, leat...",https://static.luckyscent.com/images/products/...
4,A Blvd. Called Sunset Eau de Parfum,A Lab on Fire,There's no way A Lab On Fire could relocate t...,"Bergamot, almond, violet, jasmine, leather, s...",https://static.luckyscent.com/images/products/...


In [5]:
# Etat des valeurs manquantes / vides par colonne, avant toute transformation.
# Sert à vérifier que la stratégie documentée ci-dessous reste cohérente,
# y compris si le dataset source Kaggle est mis à jour plus tard.
def compter_valeurs_vides(df):
    """Compte, par colonne, les valeurs NaN et les chaînes vides après suppression des espaces."""
    stats = {}
    for colonne in df.columns:
        nb_nan = int(df[colonne].isna().sum())
        nb_vide_non_nan = int(df[colonne].fillna("").astype(str).str.strip().eq("").sum()) - nb_nan
        stats[colonne] = {"nan": nb_nan, "vide_apres_strip": nb_vide_non_nan}
    return stats


compter_valeurs_vides(df_brut)


{'Name': {'nan': 0, 'vide_apres_strip': 0},
 'Brand': {'nan': 0, 'vide_apres_strip': 0},
 'Description': {'nan': 0, 'vide_apres_strip': 0},
 'Notes': {'nan': 80, 'vide_apres_strip': 0},
 'Image URL': {'nan': 0, 'vide_apres_strip': 0}}

## 2. Gestion des valeurs manquantes

Stratégie documentée par colonne (observée sur le dataset actuel : 0 valeur
manquante partout sauf sur `Notes`, 80 lignes soit ~3,6 %) :

| Colonne | Manquants observés | Stratégie | Justification |
|---|---|---|---|
| `Name` | 0 | Suppression de la ligne si vide | Sans nom, un parfum est inexploitable (clé d'identité) |
| `Brand` | 0 | Suppression de la ligne si vide | La marque est une donnée d'identité essentielle |
| `Description` | 0 | Suppression de la ligne si vide | Texte source du futur RAG — un parfum sans description est inutilisable pour la recherche sémantique |
| `Notes` | 80 (~3,6 %) | Conservation, `notes_list` vide + colonne `notes_manquantes=True` | Le parfum reste exploitable via sa description ; supprimer ces lignes ferait perdre des données utiles pour peu de gain |
| `Image URL` | 0 | Conservation même si vide | Non utilisée par le moteur de scoring/RAG, uniquement cosmétique côté frontend |

Le code ci-dessous reste défensif (il vérifie chaque colonne obligatoire même si
le dataset actuel n'a de valeurs manquantes que sur `Notes`), pour continuer à
fonctionner correctement si le dataset source est mis à jour.

In [6]:
COLONNES_OBLIGATOIRES = ["Name", "Brand", "Description"]


def supprimer_lignes_incompletes(df, colonnes_obligatoires):
    """
    Supprime les lignes où une colonne jugée essentielle (nom, marque,
    description) est manquante ou vide. Ces colonnes sont indispensables à
    l'identité du parfum et au futur RAG : une ligne sans elles n'est pas
    exploitable.
    """
    masque_valide = pd.Series(True, index=df.index)
    for colonne in colonnes_obligatoires:
        masque_valide &= df[colonne].notna() & (df[colonne].astype(str).str.strip() != "")
    return df[masque_valide].copy()


def marquer_notes_manquantes(df):
    """
    La colonne Notes peut être vide : on la conserve (le parfum reste
    exploitable via sa description) mais on trace explicitement l'absence via
    une colonne booléenne, utile pour le feature engineering et le RAG en aval.
    """
    df = df.copy()
    notes_vides = df["Notes"].isna() | (df["Notes"].astype(str).str.strip() == "")
    df["notes_manquantes"] = notes_vides
    return df


nb_avant = len(df_brut)
df_valide = supprimer_lignes_incompletes(df_brut, COLONNES_OBLIGATOIRES)
enregistrer_etape(
    "Suppression des lignes sans Name / Brand / Description",
    nb_avant,
    len(df_valide),
    "Colonnes jugées essentielles à l'identité du parfum",
)

df_valide = marquer_notes_manquantes(df_valide)
print(f"Lignes avec Notes manquantes conservées : {int(df_valide['notes_manquantes'].sum())}")


Lignes avec Notes manquantes conservées : 80


## 3. Normalisation des textes (casse, espaces, caractères spéciaux, encodage)

In [7]:
def normaliser_espaces(texte):
    """Supprime les espaces en début/fin et réduit les espaces multiples à un seul."""
    if pd.isna(texte):
        return texte
    return re.sub(r"\s+", " ", str(texte).strip())


def normaliser_caracteres_speciaux(texte):
    """
    Normalise l'encodage Unicode (forme NFKC, qui uniformise les variantes de
    représentation d'un même caractère accentué) et retire les caractères de
    remplacement (�) issus d'une corruption ponctuelle dans le fichier source,
    repérée sur quelques lignes isolées lors de l'exploration (notebook 01).
    """
    if pd.isna(texte):
        return texte
    texte = unicodedata.normalize("NFKC", str(texte))
    texte = texte.replace("\ufffd", "")
    return texte


def nettoyer_texte(texte):
    """Applique la normalisation complète d'un champ texte : encodage puis espaces."""
    texte = normaliser_caracteres_speciaux(texte)
    texte = normaliser_espaces(texte)
    return texte


COLONNES_TEXTE = ["Name", "Brand", "Description", "Notes"]

for colonne in COLONNES_TEXTE:
    df_valide[colonne] = df_valide[colonne].apply(nettoyer_texte)

df_valide[COLONNES_TEXTE].head()


,Name,Brand,Description,Notes
0,Tihota Eau de Parfum,Indult,"Rapa Nui for sugar, Tihota is, quite simply, T...","Vanilla bean, musks"
1,Sola Parfum,Di Ser,A tribute to the expanse of space extending fr...,"Lavender, Yuzu, Lemongrass, Magnolia, Geranium..."
2,Kagiroi Parfum,Di Ser,An aromatic ode to the ancient beauty of Japan...,"Green yuzu, green shikuwasa, sansho seed, cori..."
3,Velvet Fantasy Eau de Parfum,Montale,Velvet Fantasy is a solar fragrance where citr...,"tangerine, pink pepper, black coffee, leather,..."
4,A Blvd. Called Sunset Eau de Parfum,A Lab on Fire,There's no way A Lab On Fire could relocate to...,"Bergamot, almond, violet, jasmine, leather, sa..."


## 4. Normalisation des noms de marque

L'exploration n'a pas révélé de variantes d'écriture d'une même maison (249
marques toutes distinctes après normalisation casse/accents), seulement
quelques incohérences d'espacement sur une poignée de lignes. On garde
néanmoins une fonction dédiée et réutilisable : elle ne coûte rien aujourd'hui
et protège si le dataset source est mis à jour avec de vraies variantes.

In [8]:
def normaliser_nom_marque(marque):
    """
    Normalise un nom de marque : espaces propres (déjà fait par nettoyer_texte,
    refait ici pour que la fonction reste autonome et réutilisable). La casse
    d'origine est conservée : les marques utilisent des styles variés
    (ex. "D.S. & Durga", "MFK") qu'un .title() générique écraserait à tort.
    """
    if pd.isna(marque):
        return marque
    return re.sub(r"\s+", " ", str(marque).strip())


df_valide["Brand"] = df_valide["Brand"].apply(normaliser_nom_marque)


## 5. Suppression des doublons

Clé de déduplication : `nom + marque`, normalisés en minuscules, espaces
nettoyés et accents ignorés (pour repérer aussi d'éventuelles variantes
typographiques, ex. "Chanel N°5" vs "Chanel No 5"). Aucun doublon exact n'a
été trouvé sur le dataset actuel lors de l'exploration, mais la fonction est
conservée pour rester correcte si le dataset source évolue.

In [9]:
def construire_cle_deduplication(nom, marque):
    """
    Construit une clé de comparaison insensible à la casse, aux espaces et aux
    accents, pour détecter des doublons même en cas de légères variantes
    typographiques entre deux lignes qui désignent le même parfum.
    """
    def normaliser_pour_cle(texte):
        texte = str(texte).strip().lower()
        texte = re.sub(r"\s+", " ", texte)
        texte = unicodedata.normalize("NFKD", texte)
        return "".join(caractere for caractere in texte if not unicodedata.combining(caractere))

    return normaliser_pour_cle(nom) + "||" + normaliser_pour_cle(marque)


def supprimer_doublons(df):
    """
    Supprime les doublons sur la clé (nom + marque) normalisée, en conservant
    la première occurrence rencontrée dans le fichier source.
    """
    df = df.copy()
    df["_cle_dedup"] = df.apply(
        lambda ligne: construire_cle_deduplication(ligne["Name"], ligne["Brand"]), axis=1
    )
    df_sans_doublons = df.drop_duplicates(subset="_cle_dedup", keep="first")
    return df_sans_doublons.drop(columns="_cle_dedup")


nb_avant = len(df_valide)
df_sans_doublons = supprimer_doublons(df_valide)
enregistrer_etape(
    "Suppression des doublons (clé nom + marque normalisés)",
    nb_avant,
    len(df_sans_doublons),
    "Aucun doublon détecté sur le dataset actuel (vérifié à l'exploration) ; "
    "la fonction reste utile si le dataset source est mis à jour",
)


## 6. Parsing de la colonne `Notes` en liste (`notes_list`)

Format observé lors de l'exploration : une simple liste de notes séparées par
des virgules, en texte libre, **sans structure tête / cœur / fond**
(ex. `"Vanilla bean, musks"`). On découpe sur la virgule, on nettoie chaque
note individuellement et on la met en minuscule pour homogénéiser les futures
comparaisons/agrégations (feature engineering, notebook 03).

In [10]:
def parser_notes(texte_notes):
    """
    Transforme le texte libre de la colonne Notes en une liste Python de notes
    individuelles. Chaque note est nettoyée (espaces) et mise en minuscule
    pour homogénéiser les futures comparaisons. Les doublons internes à une
    même ligne (note répétée deux fois) sont retirés.
    """
    if pd.isna(texte_notes) or str(texte_notes).strip() == "":
        return []
    notes_brutes = str(texte_notes).split(",")
    notes_nettoyees = []
    for note in notes_brutes:
        note = normaliser_espaces(note).lower()
        if note and note not in notes_nettoyees:
            notes_nettoyees.append(note)
    return notes_nettoyees


df_sans_doublons["notes_list"] = df_sans_doublons["Notes"].apply(parser_notes)

df_sans_doublons[["Name", "Notes", "notes_list"]].head()


,Name,Notes,notes_list
0,Tihota Eau de Parfum,"Vanilla bean, musks","[vanilla bean, musks]"
1,Sola Parfum,"Lavender, Yuzu, Lemongrass, Magnolia, Geranium...","[lavender, yuzu, lemongrass, magnolia, geraniu..."
2,Kagiroi Parfum,"Green yuzu, green shikuwasa, sansho seed, cori...","[green yuzu, green shikuwasa, sansho seed, cor..."
3,Velvet Fantasy Eau de Parfum,"tangerine, pink pepper, black coffee, leather,...","[tangerine, pink pepper, black coffee, leather..."
4,A Blvd. Called Sunset Eau de Parfum,"Bergamot, almond, violet, jasmine, leather, sa...","[bergamot, almond, violet, jasmine, leather, s..."


## 7. Rapport final

In [11]:
nb_lignes_finales = len(df_sans_doublons)

print("=== Rapport de nettoyage — Nayaar ===\n")
print(f"Lignes dans le dataset brut   : {nb_lignes_brutes}")
print(f"Lignes dans le dataset propre : {nb_lignes_finales}")
print(f"Total supprimé                : {nb_lignes_brutes - nb_lignes_finales}\n")

for etape in rapport_nettoyage:
    print(
        f"- {etape['etape']} : {etape['lignes_avant']} -> {etape['lignes_apres']} "
        f"({etape['lignes_supprimees']} supprimées) — {etape['commentaire']}"
    )

print(f"\nParfums avec Notes manquantes conservés : {int(df_sans_doublons['notes_manquantes'].sum())}")


=== Rapport de nettoyage — Nayaar ===

Lignes dans le dataset brut   : 2191
Lignes dans le dataset propre : 2191
Total supprimé                : 0

- Suppression des lignes sans Name / Brand / Description : 2191 -> 2191 (0 supprimées) — Colonnes jugées essentielles à l'identité du parfum
- Suppression des doublons (clé nom + marque normalisés) : 2191 -> 2191 (0 supprimées) — Aucun doublon détecté sur le dataset actuel (vérifié à l'exploration) ; la fonction reste utile si le dataset source est mis à jour

Parfums avec Notes manquantes conservés : 80


## 8. Export vers `data/processed/`

La colonne `notes_list` est une liste Python : le CSV ne sait pas stocker ce
type nativement, elle est donc sérialisée en JSON (`json.dumps`) pour rester
lisible et re-parsable telle quelle (`json.loads`) dans les notebooks
suivants.

In [12]:
def exporter_dataset_propre(df, chemin_sortie):
    """
    Exporte le dataset nettoyé au format CSV, encodage UTF-8 (le fichier brut
    était en latin-1 avec des incohérences ; le dataset de sortie doit être
    propre et standard pour tous les traitements en aval).
    """
    df_export = df.copy()
    df_export["notes_list"] = df_export["notes_list"].apply(json.dumps)
    df_export.to_csv(chemin_sortie, index=False, encoding="utf-8")


exporter_dataset_propre(df_sans_doublons, CHEMIN_DATASET_PROPRE)
print(f"Dataset propre exporté : {CHEMIN_DATASET_PROPRE} ({len(df_sans_doublons)} lignes)")


Dataset propre exporté : ../processed/nayaar_clean.csv (2191 lignes)
